In [ ]:
# CELL 1 — SETUP & AUTOMATIC DATASET DOWNLOAD (manual fallback only if needed)
import os, sys, subprocess, zipfile, json, hashlib, sqlite3, shutil, time
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from google.colab import files
    IN_COLAB=True
except ImportError:
    IN_COLAB=False

ROOT=Path('/content' if IN_COLAB else '.').resolve()
OUT=ROOT/'Project_12_UNSW_NB15_Cybersecurity_Analytics'
(OUT/'figures').mkdir(parents=True, exist_ok=True)
(OUT/'outputs').mkdir(parents=True, exist_ok=True)

# Official dataset citation and canonical split sizes:
# https://research.unsw.edu.au/projects/unsw-nb15-dataset
# Public GitHub mirror of the original UNSW_NB15_training-set.csv / testing-set.csv.
# This is a third-party distribution mirror, not the UNSW hosting service.
DATA_BASE='https://raw.githubusercontent.com/Nir-J/ML-Projects/master/UNSW-Network_Packet_Classification/'
FALLBACK_BASE='https://raw.githubusercontent.com/jamshaid120/UNSW_NB15-Complete-dataset/main/'
REQUIRED={
    'UNSW_NB15_training-set.csv':175341,
    'UNSW_NB15_testing-set.csv':82332,
}

def looks_like_canonical_split(path, expected_rows):
    try:
        with open(path,'r',encoding='utf-8-sig',errors='replace') as handle:
            header=handle.readline().strip().lower().split(',')
            if not {'label','attack_cat','proto','service','state'}.issubset(set(header)):
                return False
            count=sum(1 for _ in handle)
        return count==expected_rows
    except (OSError, UnicodeError):
        return False

def download_verified_split(name, expected_rows):
    target=ROOT/name
    if target.exists() and looks_like_canonical_split(target,expected_rows):
        print('Already available and validated:', name)
        return True
    if target.exists():
        print('Existing file failed validation; attempting to replace:', name)
    for url in [DATA_BASE+name, FALLBACK_BASE+name]:
        temp=ROOT/(name+'.download')
        try:
            print('Downloading:', name,'from',url)
            req=Request(url,headers={'User-Agent':'Mozilla/5.0 Project12-Research/1.0'})
            with urlopen(req,timeout=55) as response, open(temp,'wb') as out:
                shutil.copyfileobj(response,out)
            if not looks_like_canonical_split(temp,expected_rows):
                raise ValueError('unexpected header or record count; not a canonical split')
            temp.replace(target)
            print('Validated:',name,'| records:',expected_rows,'| SHA-256:',hashlib.sha256(target.read_bytes()).hexdigest())
            return True
        except Exception as exc:
            print('Download source unavailable or validation failed:',str(exc)[:230])
            temp.unlink(missing_ok=True)
    return False

missing=[name for name,count in REQUIRED.items() if not download_verified_split(name,count)]
if missing:
    # A manual upload may also contain a ZIP with the official files.
    if IN_COLAB:
        print('Automatic download unavailable for:', missing)
        print('Please upload the canonical CSV files or an archive containing them.')
        print('Official source: https://research.unsw.edu.au/projects/unsw-nb15-dataset')
        files.upload()
    for z in ROOT.glob('*.zip'):
        try:
            with zipfile.ZipFile(z) as obj:
                for item in obj.namelist():
                    name=Path(item).name
                    if name in REQUIRED and (not (ROOT/name).exists() or not looks_like_canonical_split(ROOT/name,REQUIRED[name])):
                        temp=ROOT/(name+'.extracted')
                        with obj.open(item) as src, open(temp,'wb') as dest:
                            shutil.copyfileobj(src,dest)
                        if looks_like_canonical_split(temp,REQUIRED[name]):
                            temp.replace(ROOT/name)
                        else:
                            temp.unlink(missing_ok=True)
        except zipfile.BadZipFile:
            print('Skipping invalid ZIP:',z.name)
    missing=[name for name,count in REQUIRED.items() if not (ROOT/name).exists() or not looks_like_canonical_split(ROOT/name,count)]
    if missing:
        raise FileNotFoundError('Required validated split files still missing: '+', '.join(missing))

print('Data ready — no manual upload needed when automatic downloads succeed.')
print('Working directory:',OUT)
print('PROVENANCE: UNSW-NB15 academic dataset; third-party public GitHub mirrors for retrieval.')
print('SCOPE: Historical academic benchmark, not live network monitoring.')


In [ ]:
# CELL 2 — DISCOVER & VALIDATE CANONICAL CSV SPLITS
paths=list(ROOT.rglob('*.csv'))
tr_candidates=[p for p in paths if 'training-set' in p.name.lower() and 'unsw' in p.name.lower()]
te_candidates=[p for p in paths if 'testing-set' in p.name.lower() and 'unsw' in p.name.lower()]
assert tr_candidates and te_candidates, ('Missing official split CSV files. Required: UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv. Found: '+str([x.name for x in paths[:30]]))
train=pd.read_csv(tr_candidates[0],low_memory=False)
test=pd.read_csv(te_candidates[0],low_memory=False)
train.columns=train.columns.str.strip().str.lower()
test.columns=test.columns.str.strip().str.lower()
assert train.columns.tolist()==test.columns.tolist(),'Training and test columns do not match; obtain canonical files.'
assert {'label','attack_cat','proto','service','state'}.issubset(train.columns), f'Unexpected schema: {train.columns.tolist()}'
for frame in (train,test):
    frame['label']=pd.to_numeric(frame['label'],errors='raise').astype(int)
    assert set(frame['label'].dropna().unique()).issubset({0,1}), 'label must be 0 / 1'
    frame['attack_cat']=frame['attack_cat'].fillna('Normal').astype(str).str.strip().replace({'':'Normal'})
    frame.loc[frame['label'].eq(0),'attack_cat']='Normal'
assert len(train)>100000 and len(test)>50000,'These do not look like full official split files.'
print('TRAIN:',train.shape,'TEST:',test.shape,'Columns:',len(train.columns))
print('Dataset periods are historical research records, NOT live network traffic.')

In [ ]:
# CELL 3 — DATA QUALITY, ATTACK-CATEGORY AUDIT & OUTPUT FILES
parts={'train':train,'test':test}
rows=[]
for split,frame in parts.items():
    rows.append(dict(split=split,records=len(frame),columns=len(frame.columns),missing_cells=int(frame.isna().sum().sum()),duplicate_rows=int(frame.duplicated().sum()),normal_records=int((frame.label==0).sum()),attack_records=int((frame.label==1).sum()),distinct_attack_categories=int(frame.attack_cat.nunique())))
quality=pd.DataFrame(rows)
quality.to_csv(OUT/'outputs/data_quality_audit.csv',index=False)
cat=(pd.concat([train.assign(split='train'),test.assign(split='test')],ignore_index=True)
     .groupby(['split','attack_cat'],dropna=False).size().reset_index(name='records'))
cat['share_pct']=cat['records']/cat.groupby('split')['records'].transform('sum')*100
cat.to_csv(OUT/'outputs/attack_category_by_split.csv',index=False)
assert (quality.normal_records+quality.attack_records==quality.records).all()
print(quality.to_string(index=False))
print(cat.sort_values(['split','records'],ascending=[True,False]).to_string(index=False))

In [ ]:
# CELL 4 — SQLITE ANALYTICS & RECONCILIATION
con=sqlite3.connect(':memory:')
for split,frame in parts.items():
    frame[['label','attack_cat','proto','service','state']].assign(split=split).to_sql('network_flows',con,if_exists='append',index=False)
q1=pd.read_sql_query('SELECT split,proto, COUNT(*) AS records, SUM(label) AS attacks FROM network_flows GROUP BY split,proto ORDER BY split,records DESC',con)
q2=pd.read_sql_query('SELECT split,service, COUNT(*) AS records, SUM(label) AS attacks FROM network_flows GROUP BY split,service ORDER BY split,records DESC',con)
q3=pd.read_sql_query('SELECT split,state, COUNT(*) AS records, SUM(label) AS attacks FROM network_flows GROUP BY split,state ORDER BY split,records DESC',con)
for name,frame in [('protocol_breakdown',q1),('service_breakdown',q2),('connection_state_breakdown',q3)]:
    frame.to_csv(OUT/f'outputs/{name}.csv',index=False)
    for split,df in parts.items():
        assert int(frame.loc[frame['split'].eq(split),'records'].sum())==len(df),name+' reconciliation failed'
print('SQL reconciled to both full partitions. Top TEST services:')
print(q2[q2['split'].eq('test')].head(10).to_string(index=False))

In [ ]:
# CELL 5 — HELD-OUT BASELINE CLASSIFIER (NO TEST-DATA FITTING)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score
excluded={'id','label','attack_cat'}
cols=[c for c in train.columns if c not in excluded]
catcols=[c for c in cols if train[c].dtype=='object' or str(train[c].dtype)=='category']
numcols=[c for c in cols if c not in catcols]
Xtr=train[cols].copy(); Xte=test[cols].copy()
for col in numcols:
    Xtr[col]=pd.to_numeric(Xtr[col],errors='coerce')
    Xte[col]=pd.to_numeric(Xte[col],errors='coerce')
for col in catcols:
    Xtr[col]=Xtr[col].astype('string').fillna('unknown').astype(str)
    Xte[col]=Xte[col].astype('string').fillna('unknown').astype(str)
pre=ColumnTransformer([('num',SimpleImputer(strategy='median'),numcols),('category',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),catcols)])
model=Pipeline([('preprocessing',pre),('classifier',HistGradientBoostingClassifier(max_iter=110,max_leaf_nodes=25,learning_rate=.08,random_state=42))])
model.fit(Xtr,train.label)
pred=model.predict(Xte)
tn,fp,fn,tp=map(int,confusion_matrix(test.label,pred,labels=[0,1]).ravel())
metrics={'train_records':len(train),'test_records':len(test),'train_columns':len(train.columns),'test_columns':len(test.columns),'model':'HistGradientBoostingClassifier','split':'official UNSW-NB15 train/test partitions','accuracy':float(accuracy_score(test.label,pred)),'balanced_accuracy':float(balanced_accuracy_score(test.label,pred)),'attack_precision':float(precision_score(test.label,pred,zero_division=0)),'attack_recall':float(recall_score(test.label,pred,zero_division=0)),'attack_f1':float(f1_score(test.label,pred,zero_division=0)),'true_negative':tn,'false_positive':fp,'false_negative':fn,'true_positive':tp,'false_positive_rate':fp/(fp+tn),'miss_rate':fn/(fn+tp),'features_used':len(cols),'test_used_only_for_evaluation':True,'scope':'historical academic benchmark; not a production SOC'}
(OUT/'outputs/holdout_model_metrics.json').write_text(json.dumps(metrics,indent=2))
errors=test[['attack_cat','label']].copy();errors['predicted_attack']=pred
by_category=errors.groupby('attack_cat').agg(records=('label','size'),attacks=('label','sum'),flagged=('predicted_attack','sum')).reset_index()
by_category['true_positives']=errors[(errors.label==1)&(errors.predicted_attack==1)].groupby('attack_cat').size().reindex(by_category.attack_cat,fill_value=0).values
by_category['missed_attacks']=by_category['attacks']-by_category['true_positives']
by_category.to_csv(OUT/'outputs/category_detection_performance.csv',index=False)
assert tn+fp+fn+tp==len(test)
print(json.dumps(metrics,indent=2))

In [ ]:
# CELL 6 — SOC-STYLE STATIC DASHBOARD
plt.rcParams.update({'figure.dpi':130,'font.size':10})
fig,axes=plt.subplots(2,2,figsize=(15,10),constrained_layout=True)
a=axes[0,0]
ct=test['attack_cat'].value_counts().head(10).sort_values()
a.barh(ct.index,ct.values);a.set_title('Held-out test category composition');a.set_xlabel('Records (not incident frequency)')
a=axes[0,1]
a.bar(['Normal','Attack'],[(test.label==0).sum(),(test.label==1).sum()]);a.set_title('Test labels');a.set_ylabel('Records')
a=axes[1,0]
a.bar(['Precision','Recall','F1'],[metrics['attack_precision'],metrics['attack_recall'],metrics['attack_f1']]);a.set_ylim(0,1);a.set_title('Baseline held-out classification metrics')
a=axes[1,1]
a.imshow([[tn,fp],[fn,tp]],cmap='Blues')
for i,row in enumerate([[tn,fp],[fn,tp]]):
    for j,val in enumerate(row):a.text(j,i,f'{val:,}',ha='center',va='center',fontweight='bold')
a.set_xticks([0,1],['Predicted normal','Predicted attack']);a.set_yticks([0,1],['True normal','True attack']);a.set_title('Held-out confusion matrix')
fig.suptitle('PROJECT 12 | UNSW-NB15 Defensive Intrusion Analytics\nHistorical academic benchmark — NOT live alerts or current threats',fontsize=17,fontweight='bold')
fig.savefig(OUT/'figures/soc_executive_dashboard.png',bbox_inches='tight',dpi=160)
plt.show()
print('Dashboard saved:',OUT/'figures/soc_executive_dashboard.png')

In [ ]:
# CELL 7 — EVIDENCE-BASED RECOMMENDATIONS & REPRODUCIBLE DOCUMENTATION
rec=[
(1,'Report evaluation honestly',f'Test attack recall {metrics["attack_recall"]:.1%}; {fn:,} false negatives','Prioritize recall and review missed attack categories alongside precision','Track held-out precision, recall, F1 and miss rate'),
(2,'Investigate false alerts',f'{fp:,} false positives; FPR {metrics["false_positive_rate"]:.1%}','Examine normal traffic incorrectly flagged in a controlled validation sample','Monitor false-positive rate and analyst-review burden'),
(3,'Investigate category-level blind spots','Per-category held-out errors exported','Review underperforming categories and imbalance before choosing model changes','Report attack-category recall and sample support'),
(4,'Preserve leakage-safe evaluation','Official training and held-out testing partitions are separate','Fit encoders, imputers and model only on training partition','No preprocessing fit on test; repeatable metrics'),
(5,'Strengthen source data validation',f'{len(train):,} train rows; {len(test):,} test rows','Monitor missingness, duplicates, schema and numeric conversion failures','Schema passes and audit CSVs reconcile'),
(6,'Use SQL segmentation for triage',f'{test.attack_cat.nunique()} categories including Normal in test','Review patterns by service, protocol, state and category without inventing attack chronology','SQL groups reconcile to total test records'),
(7,'Validate on relevant modern data before operational use','UNSW-NB15 is a historical research benchmark','Obtain authorized recent datasets and evaluate drift, deployment risks and privacy','Never claim live SOC capability without separate validation')]
recommendations=pd.DataFrame(rec,columns=['priority','recommendation','evidence','action','success_measure'])
recommendations.to_csv(OUT/'outputs/recommendations.csv',index=False)
(OUT/'requirements.txt').write_text('numpy\npandas\nmatplotlib\nscikit-learn\nnbformat\n')
(OUT/'.gitignore').write_text('*.zip\n__pycache__/\n.ipynb_checkpoints/\n.DS_Store\n')
(OUT/'LICENSE_DATA_NOTICE.md').write_text('UNSW-NB15 originates from UNSW Canberra. Verify the upstream redistribution rights before publishing raw CSV files. This package excludes source raw records. Cite UNSW researchers and research papers.\n')
readme=f'''# UNSW-NB15 Network Intrusion & SOC Analytics — Project 12

## Executive summary
Defensive SQL, Python and machine-learning analysis of the **UNSW-NB15 historical academic network intrusion benchmark**. **NOT real-time SOC monitoring**. No time trend, current threat claims, or deployment guarantee.

## Sources and provenance
- [UNSW-NB15 official research description](https://research.unsw.edu.au/projects/unsw-nb15-dataset)
- Dataset files required: `UNSW_NB15_training-set.csv`, `UNSW_NB15_testing-set.csv`.
- Official training/testing sizes: {len(train):,} / {len(test):,} rows (observed).
- The academic data represents research traffic, including simulated attack behaviors; avoid treating proportions as real-world incident prevalence.
- Do not redistribute source records without confirming licensing and permitted reuse.

## Methodology
1. Import canonical train/test partitions; validate headers and labels.
2. Audit missing fields, duplicates and attack categories.
3. Run SQLite segmentation by protocol, service and connection state; reconcile counts.
4. Train a baseline HistGradientBoostingClassifier with train-only imputation/encoding.
5. Score official untouched test partition; inspect category detection errors.
6. Generate a reproducible PNG dashboard and evidence-based recommendations.

## Held-out benchmark metrics
| Measure | Value |
|---|---:|
| Training rows | {len(train):,} |
| Testing rows | {len(test):,} |
| Attack precision | {metrics['attack_precision']:.4f} |
| Attack recall | {metrics['attack_recall']:.4f} |
| Attack F1 | {metrics['attack_f1']:.4f} |
| Accuracy | {metrics['accuracy']:.4f} |
| False positives | {fp:,} |
| False negatives | {fn:,} |

## SOC-style static dashboard
![Historical UNSW-NB15 SOC-style executive dashboard](figures/soc_executive_dashboard.png)

## Recommendations
'''
for _,r in recommendations.iterrows():readme+=f"{int(r.priority)}. **{r.recommendation}** — {r.evidence}. Action: {r.action}. KPI: {r.success_measure}.\n"
readme+='''
## Project outputs
- `outputs/data_quality_audit.csv`: train/test data integrity
- `outputs/attack_category_by_split.csv`: category distributions
- `outputs/protocol_breakdown.csv`, `service_breakdown.csv`, `connection_state_breakdown.csv`: reconciled SQL
- `outputs/holdout_model_metrics.json`: benchmark confusion matrix and model metrics
- `outputs/category_detection_performance.csv`: category-specific holdout model gaps
- `outputs/recommendations.csv`: seven actionable recommendations

## Limitations
- No verified wall-clock alert timestamps, endpoint identities, organizations or geographic incidents in the canonical training/testing features.
- These benchmark model metrics do not establish production reliability or real SOC incident-detection effectiveness.
- Only use data and cloud environments with legitimate authorization.

## Reproduce on Google Colab
Open `Project_12_UNSW_NB15_SOC_Analytics.ipynb`; run all eight code cells; upload the two official split CSV files or a ZIP containing them. The last cell downloads a ZIP with the deliverables. Upload the saved notebook and extracted project files (not just the ZIP) to GitHub.
'''
(OUT/'README.md').write_text(readme,encoding='utf-8')
print('Recommendations:',len(recommendations),'README and package metadata generated')

In [ ]:
# CELL 8 — VALIDATE FILES, GENERATE MANIFEST & DOWNLOAD GITHUB ZIP
required=['README.md','requirements.txt','.gitignore','LICENSE_DATA_NOTICE.md','figures/soc_executive_dashboard.png','outputs/data_quality_audit.csv','outputs/attack_category_by_split.csv','outputs/protocol_breakdown.csv','outputs/service_breakdown.csv','outputs/connection_state_breakdown.csv','outputs/holdout_model_metrics.json','outputs/category_detection_performance.csv','outputs/recommendations.csv']
for rel in required:
    assert (OUT/rel).is_file() and (OUT/rel).stat().st_size>0,f'Missing/empty: {rel}'
assert len(pd.read_csv(OUT/'outputs/recommendations.csv'))==7
assert int(json.loads((OUT/'outputs/holdout_model_metrics.json').read_text())['test_records'])==len(test)
(OUT/'PROJECT_STATUS.md').write_text('STATUS: LOCALLY GENERATED — upload to GitHub and verify before FINAL PASS.\\n')
manifest=[]
for f in sorted(OUT.rglob('*')):
    if f.is_file() and f.name!='FILE_MANIFEST.csv':manifest.append(dict(file=f.relative_to(OUT).as_posix(),bytes=f.stat().st_size,sha256=hashlib.sha256(f.read_bytes()).hexdigest()))
pd.DataFrame(manifest).to_csv(OUT/'FILE_MANIFEST.csv',index=False)
archive=ROOT/'Project_12_UNSW_NB15_SOC_GitHub_Package.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file():z.write(f,f.relative_to(OUT))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    print('ZIP contents:');print('\\n'.join(z.namelist()))
print('Verified',len(manifest),'artifacts; ZIP:',archive,'size MB:',round(archive.stat().st_size/1e6,2))
print('NOTE: Save the notebook separately and upload it to GitHub with the extracted files.')
if IN_COLAB:files.download(str(archive))